In [0]:
# Databricks notebook source

CATALOG = "github_pipeline_catalog"
SCHEMA = "github_analytics"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS "
    f"`{CATALOG}`.{SCHEMA}.raw_data"
)

RAW_VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"

print(RAW_VOLUME)

In [0]:

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA `{SCHEMA}`")

print("Catalog:", CATALOG)
print("Schema:", SCHEMA)

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`pipeline_execution_logs`
(
    run_id STRING,
    pipeline_name STRING,
    layer STRING,
    dataset STRING,
    batch_id STRING,
    input_parameter STRING,
    source_load_type STRING,
    execution_mode STRING,
    execution_start TIMESTAMP,
    execution_end TIMESTAMP,
    status STRING,
    rows_read BIGINT,
    rows_inserted BIGINT,
    rows_updated BIGINT,
    rows_quarantined BIGINT,
    error_message STRING
)
USING DELTA
""")

print("pipeline_execution_logs created")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`quarantine_records`
(
    dataset STRING,
    layer STRING,
    batch_id STRING,
    source_file STRING,
    business_key STRING,
    error_reason STRING,
    record_json STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

print("quarantine_records created")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`bronze_repositories`
(
    repo_id STRING,
    source_repository STRING,
    name STRING,
    full_name STRING,
    owner_login STRING,
    description STRING,
    language STRING,
    stargazers_count STRING,
    forks_count STRING,
    open_issues_count STRING,
    default_branch STRING,
    archived BOOLEAN,
    disabled BOOLEAN,
    fork BOOLEAN,
    created_at_raw STRING,
    updated_at_raw STRING,
    pushed_at_raw STRING,
    fetched_at_utc_raw STRING,
    batch_id STRING,
    source_load_type STRING,
    execution_mode STRING,
    source_file STRING,
    schema_version STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`bronze_commits`
(
    commit_sha STRING,
    source_repository STRING,
    author_login STRING,
    author_name STRING,
    author_email STRING,
    author_date_raw STRING,
    committer_login STRING,
    committer_name STRING,
    committer_email STRING,
    committer_date_raw STRING,
    message STRING,
    fetched_at_utc_raw STRING,
    batch_id STRING,
    source_load_type STRING,
    execution_mode STRING,
    source_file STRING,
    schema_version STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`bronze_issues`
(
    issue_id STRING,
    issue_number STRING,
    source_repository STRING,
    state STRING,
    state_reason STRING,
    title STRING,
    user_login STRING,
    comments_raw STRING,
    created_at_raw STRING,
    updated_at_raw STRING,
    closed_at_raw STRING,
    is_pull_request BOOLEAN,
    fetched_at_utc_raw STRING,
    batch_id STRING,
    source_load_type STRING,
    execution_mode STRING,
    source_file STRING,
    schema_version STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`bronze_pull_requests`
(
    pr_id STRING,
    pr_number STRING,
    source_repository STRING,
    state STRING,
    title STRING,
    user_login STRING,
    draft BOOLEAN,
    created_at_raw STRING,
    updated_at_raw STRING,
    closed_at_raw STRING,
    merged_at_raw STRING,
    merge_commit_sha STRING,
    fetched_at_utc_raw STRING,
    batch_id STRING,
    source_load_type STRING,
    execution_mode STRING,
    source_file STRING,
    schema_version STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

## Silver Tables

### silver_repositories

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`silver_repositories`
(
    repo_id BIGINT,
    source_repository STRING,
    name STRING,
    full_name STRING,
    owner_login STRING,
    description STRING,
    language STRING,
    stars BIGINT,
    forks BIGINT,
    open_issues BIGINT,
    default_branch STRING,
    archived BOOLEAN,
    disabled BOOLEAN,
    fork BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    pushed_at TIMESTAMP,
    record_hash STRING,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

### silver commits

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`silver_commits`
(
    commit_sha STRING,
    repo_id BIGINT,
    source_repository STRING,
    author_login_hash STRING,
    author_email_hash STRING,
    author_date TIMESTAMP,
    committer_date TIMESTAMP,
    message STRING,
    record_hash STRING,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

### silver issues

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`silver_issues`
(
    issue_id BIGINT,
    repo_id BIGINT,
    source_repository STRING,
    issue_number BIGINT,
    state STRING,
    state_reason STRING,
    title STRING,
    user_login_hash STRING,
    comments BIGINT,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    closed_at TIMESTAMP,
    record_hash STRING,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

### silver pull request

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`.`silver_pull_requests`
(
    pr_id BIGINT,
    repo_id BIGINT,
    source_repository STRING,
    pr_number BIGINT,
    state STRING,
    title STRING,
    user_login_hash STRING,
    draft BOOLEAN,
    created_at TIMESTAMP,
    updated_at TIMESTAMP,
    closed_at TIMESTAMP,
    merged_at TIMESTAMP,
    merge_commit_sha STRING,
    record_hash STRING,
    source_batch_id STRING,
    load_timestamp TIMESTAMP
)
USING DELTA
""")

### Verifying all the tables

In [0]:
display(
    spark.sql(
        f"SHOW TABLES IN `{CATALOG}`.`{SCHEMA}`"
    )
)